# Differentiable radar/drag profile training

Reusable geometry, radar, flow-solver, convergence, and persistence code lives in `radar_helpers.py`. This notebook contains only configuration and training orchestration.


In [1]:
import json
import math
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

# Support kernels started from either the repository root or this folder.
RADAR_DIR = Path.cwd() / "radar"
if not (RADAR_DIR / "radar_helpers.py").exists():
    RADAR_DIR = Path.cwd()
if str(RADAR_DIR) not in sys.path:
    sys.path.insert(0, str(RADAR_DIR))

from radar_helpers import (
    MRTExternalFlow,
    append_history_csv,
    circle_vertices,
    disk_reference,
    implicit_steady_drag,
    initial_parameters,
    monostatic_q,
    polygon_area,
    polygon_fourier_power,
    profile_mask,
    settle_to_steady,
)


In [2]:
# Edit these constants directly before running the training cells.
cfg = {
    "device": "cuda",
    "outdir": "runs/mrt_rcs_sharp",
    "radar_weight": 0.0,
    "cells_per_diameter": 96,
    "reynolds": 40.0,
    "inflow_speed": 1.0 / 30.0,
    "target_area": 0.205,
    "domain_x_d": [-4.0, 10.0],
    "domain_y_d": [-5.0, 5.0],
    "grid_offsets": [[0.0, 0.0], [0.5, 0.5]],
    "interface_cells": 1.0,
    "optimizer_steps": 150,
    "minimum_optimizer_steps": 60,
    "learning_rate": 5e-3,
    "adjoint_fixed_point_steps": 16,
    "adjoint_restart": 64,
    "adjoint_max_iterations": 224,
    "adjoint_relative_tolerance": 7e-3,
    "adjoint_absolute_tolerance": 1e-7,
    "adjoint_primal_relative_tolerance": 5e-3,
    "gradient_clip": 2.0,
    "modes_count": 8,
    "initial_chord": 0.70,
    "settle_min_steps": 400,
    "settle_max_steps_per_update": 3000,
    "initial_settle_max_steps": 8000,
    "reference_settle_max_steps": 24000,
    "settle_chunk_steps": 200,
    "settle_drag_tolerance": 5e-4,
    "settle_velocity_tolerance": 2e-3,
    "reference_drag_tolerance": 5e-4,
    "reference_velocity_tolerance": 2e-3,
    "average_windows": 6,
    "average_window_steps": 200,
    "convergence_window": 15,
    "optimization_loss_tolerance": 2e-4,
    "optimization_grad_tolerance": 2e-3,
    "optimization_patience": 3,
    "radar_angles": 192,
    "radar_wavelengths": 13,
    "wavelength_min": 0.18,
    "wavelength_max": 0.24,
    "debug_every": 10,
    "seed": 21,
    "compile_step": True,
    "compile_mode": "default",
    "ignore_disk_cache": False,
    "require_settled": True,
    "max_mach": 0.15,
    "max_mass_error": 1e-3,
}


## Solver setup


In [3]:
outdir = Path(cfg["outdir"])
outdir.mkdir(parents=True, exist_ok=True)
(outdir / "config.json").write_text(json.dumps(cfg, indent=2))

device = torch.device(cfg["device"])
if device.type != "cuda":
    raise ValueError("This training configuration is intended for CUDA.")

torch.manual_seed(cfg["seed"])
torch.cuda.manual_seed_all(cfg["seed"])
torch.set_float32_matmul_precision("highest")
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

solver = MRTExternalFlow(
    cells_per_diameter=cfg["cells_per_diameter"],
    reynolds=cfg["reynolds"],
    inflow_speed=cfg["inflow_speed"],
    reference_area=cfg["target_area"],
    domain_x_d=cfg["domain_x_d"],
    domain_y_d=cfg["domain_y_d"],
    device=device,
    dtype=torch.float32,
    compile_step=cfg["compile_step"],
    compile_mode=cfg["compile_mode"],
)


## Disk reference values


In [4]:
print(
    f"grid={solver.nx}x{solver.ny}, D={solver.cells_per_diameter} cells, "
    f"Re={solver.reynolds:g}, Ma={solver.inflow_speed / solver.cs:.4f}, "
    f"tau_shear={solver.tau_shear:.4f}"
)
print(
    f"boundary={solver.boundary_scheme}, domain={solver.domain_x_d}D x "
    f"{solver.domain_y_d}D, grid offsets={cfg['grid_offsets']}"
)

disk_vertices_value = circle_vertices(
    device, torch.float32, target_area=cfg["target_area"]
)
radar_angles = torch.arange(
    cfg["radar_angles"], device=device, dtype=torch.float32
) * (2.0 * math.pi / cfg["radar_angles"])
radar_wavelengths = torch.linspace(
    cfg["wavelength_min"],
    cfg["wavelength_max"],
    cfg["radar_wavelengths"],
    device=device,
    dtype=torch.float32,
)
q = monostatic_q(
    radar_wavelengths, radar_angles, device, torch.float32
)

disk_area = polygon_area(disk_vertices_value)
disk_rcs = (
    polygon_fourier_power(disk_vertices_value, q)
    / disk_area[:, None].square()
).mean().detach()

disk_state, disk_cd, disk_cd_std = disk_reference(solver, cfg, outdir)
print(
    f"disk Cd={disk_cd.item():.6f} +/- {disk_cd_std.item():.2e}"
)


grid=1344x960, D=96 cells, Re=40, Ma=0.0577, tau_shear=0.7400
boundary=sharp_ste_halfway_v1, domain=(-4.0, 10.0)D x (-5.0, 5.0)D, grid offsets=[[0.0, 0.0], [0.5, 0.5]]
disk Cd=2.029617 +/- 4.17e-03


## Trainable profile and optimizer


In [5]:
params = torch.nn.Parameter(
    initial_parameters(
        device,
        torch.float32,
        cfg["modes_count"],
        chord=cfg["initial_chord"],
    )
)
optimizer = torch.optim.AdamW(
    [params], lr=cfg["learning_rate"], weight_decay=0.0
)

history = []
initial_flow_ready = False
# Fully establish the initial flow before taking any geometry gradient.
with torch.no_grad():
    initial_mask, _, _, _ = profile_mask(
        params,
        solver,
        target_area=cfg["target_area"],
        interface_cells=cfg["interface_cells"],
        grid_offsets=cfg["grid_offsets"],
    )
    # A converged circle wake is a much better initial condition than uniform flow.
    flow_state = disk_state.detach().clone()
    flow_state, initial_settle_diag = settle_to_steady(
        solver,
        flow_state,
        initial_mask,
        min_steps=cfg["settle_min_steps"],
        max_steps=cfg["initial_settle_max_steps"],
        chunk_steps=cfg["settle_chunk_steps"],
        drag_tolerance=cfg["reference_drag_tolerance"],
        velocity_tolerance=cfg["reference_velocity_tolerance"],
        report_every=1000,
        label="initial profile",
    )

solver.validate_diagnostics(
    initial_settle_diag,
    max_mach_limit=cfg["max_mach"],
    mass_error_limit=cfg["max_mass_error"],
)
if cfg["require_settled"] and not initial_settle_diag["settled"]:
    raise RuntimeError(
        "Initial profile flow did not settle within "
        f"{initial_settle_diag['settle_steps']} steps."
    )
print(
    f"initial flow settled in {initial_settle_diag['settle_steps']} steps; "
    f"Ma_max={initial_settle_diag['max_mach'].max().item():.4f}, "
    f"density_error={initial_settle_diag['mass_error'].max().item():.2e}"
)
flow_params = params.detach().clone()
last_settle_diag = initial_settle_diag
initial_flow_ready = True

loss_window = []
convergence_hits = 0
history_csv = outdir / "history.csv"


initial profile: steps=1000, Cd=1.815423, dCd=1.25e-03, du=4.64e-03
initial profile: steps=2000, Cd=1.733985, dCd=5.01e-03, du=1.66e-03
initial profile: steps=3000, Cd=1.711073, dCd=7.67e-04, du=1.46e-03
initial profile: steps=4000, Cd=1.700293, dCd=4.22e-04, du=1.36e-03
initial flow settled in 4000 steps; Ma_max=0.0763, density_error=7.95e-05


## Optimization loop


In [6]:
from tqdm import tqdm
if not globals().get("initial_flow_ready", False):
    raise RuntimeError(
        "Run the optimizer setup cell to completion before training."
    )
for step in tqdm(range(cfg["optimizer_steps"])):
    wall_start = time.perf_counter()
    optimizer.zero_grad(set_to_none=True)

    mask, vertices, chord, _ = profile_mask(
        params,
        solver,
        target_area=cfg["target_area"],
        interface_cells=cfg["interface_cells"],
        grid_offsets=cfg["grid_offsets"],
    )

    # Re-equilibrate only when parameters have changed. Step zero reuses
    # the fully settled state prepared by the optimizer setup cell.
    geometry_changed = not torch.equal(params.detach(), flow_params)
    if geometry_changed:
        flow_state, settle_diag = settle_to_steady(
            solver,
            flow_state.detach(),
            mask.detach(),
            min_steps=cfg["settle_min_steps"],
            max_steps=cfg["settle_max_steps_per_update"],
            chunk_steps=cfg["settle_chunk_steps"],
            drag_tolerance=cfg["settle_drag_tolerance"],
            velocity_tolerance=cfg["settle_velocity_tolerance"],
        )
        if settle_diag["settled"]:
            flow_params = params.detach().clone()
            last_settle_diag = settle_diag
    else:
        settle_diag = last_settle_diag

    solver.validate_diagnostics(
        settle_diag,
        max_mach_limit=cfg["max_mach"],
        mass_error_limit=cfg["max_mass_error"],
    )
    if cfg["require_settled"] and not settle_diag["settled"]:
        raise RuntimeError(
            f"Flow did not settle at optimizer step {step} within "
            f"{settle_diag['settle_steps']} steps."
        )

    cd = implicit_steady_drag(
        solver,
        flow_state.detach(),
        mask,
        fixed_point_steps=cfg["adjoint_fixed_point_steps"],
        restart=cfg["adjoint_restart"],
        max_iterations=cfg["adjoint_max_iterations"],
        relative_tolerance=cfg["adjoint_relative_tolerance"],
        absolute_tolerance=cfg["adjoint_absolute_tolerance"],
        primal_relative_tolerance=cfg["adjoint_primal_relative_tolerance"],
    )

    area = polygon_area(vertices)
    rcs = (
        polygon_fourier_power(vertices, q)
        / area[:, None].square()
    ).mean(dim=1)

    relative_cd = cd / disk_cd
    relative_rcs = rcs / disk_rcs

    loss = (
        relative_cd
        + cfg["radar_weight"] * relative_rcs
    ).mean()

    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(
        [params], cfg["gradient_clip"]
    )
    optimizer.step()
    adjoint_info = solver.last_adjoint_info

    torch.cuda.synchronize(device)
    wall_seconds = time.perf_counter() - wall_start

    record = {
        "step": step,
        "loss": float(loss.detach().cpu()),
        "relative_cd": float(relative_cd.mean().detach().cpu()),
        "relative_rcs": float(relative_rcs.mean().detach().cpu()),
        "cd": float(cd.mean().detach().cpu()),
        "rcs": float(rcs.mean().detach().cpu()),
        "area": float(area.mean().detach().cpu()),
        "chord": float(chord.mean().detach().cpu()),
        "grad_norm": float(torch.as_tensor(grad_norm).detach().cpu()),
        "adjoint_iterations": adjoint_info["iterations"],
        "adjoint_relative_residual": adjoint_info["relative_residual"],
        "primal_fixed_point_residual": solver.last_primal_fixed_point_residual,
        "settle_steps": settle_diag["settle_steps"],
        "settled": bool(settle_diag["settled"]),
        "settle_drag_change": settle_diag["drag_relative_change"],
        "settle_velocity_change": settle_diag["velocity_relative_change"],
        "max_mach": float(settle_diag["max_mach"].max().cpu()),
        "mass_error": float(settle_diag["mass_error"].max().cpu()),
        "wall_seconds": wall_seconds,
    }
    history.append(record)
    append_history_csv(history_csv, record)

    print(
        f"step={step:04d} loss={record['loss']:.6f} "
        f"Cd/disk={record['relative_cd']:.5f} "
        f"RCS/disk={record['relative_rcs']:.5f} "
        f"chord={record['chord']:.4f} grad={record['grad_norm']:.3e} "
        f"adj={record['adjoint_iterations']} "
        f"r_adj={record['adjoint_relative_residual']:.2e} "
        f"r_flow={record['primal_fixed_point_residual']:.2e} "
        f"settle={record['settle_steps']} "
        f"du={record['settle_velocity_change']:.2e} "
        f"dCd={record['settle_drag_change']:.2e} "
        f"Ma_max={record['max_mach']:.3f} "
        f"time={wall_seconds:.1f}s"
    )

    if (step + 1) % cfg["debug_every"] == 0:
        with torch.no_grad():
            debug_mask, debug_vertices, _, _ = profile_mask(
                params,
                solver,
                target_area=cfg["target_area"],
                interface_cells=cfg["interface_cells"],
                grid_offsets=cfg["grid_offsets"],
            )

        shape_xy = debug_vertices[0].detach().cpu().numpy()
        closed_shape = np.vstack((shape_xy, shape_xy[0]))
        mask_image = debug_mask[0].detach().cpu().numpy()

        fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
        axes[0].fill(shape_xy[:, 0], shape_xy[:, 1], alpha=0.35)
        axes[0].plot(closed_shape[:, 0], closed_shape[:, 1], linewidth=2)
        axes[0].set_title("100-vertex profile")
        axes[0].set_aspect("equal")
        axes[0].set_xlim(-0.55, 0.55)
        axes[0].set_ylim(-0.55, 0.55)
        axes[0].grid(alpha=0.25)

        axes[1].imshow(
            mask_image,
            origin="lower",
            extent=(solver.xmin, solver.xmax, solver.ymin, solver.ymax),
            cmap="magma",
            vmin=0.0,
            vmax=1.0,
        )
        axes[1].set_title("Differentiable solver mask")
        axes[1].set_aspect("equal")
        axes[1].set_xlabel("x")
        axes[1].set_ylabel("y")

        fig.suptitle(
            f"Optimizer step {step + 1} | "
            f"loss={record['loss']:.5f}, Cd/disk={record['relative_cd']:.4f}, "
            f"RCS/disk={record['relative_rcs']:.4f}"
        )
        plt.show()
        plt.close(fig)

    loss_window.append(record["loss"])
    if len(loss_window) > cfg["convergence_window"]:
        loss_window.pop(0)

    if step >= cfg["minimum_optimizer_steps"] and len(loss_window) == cfg["convergence_window"]:
        relative_span = (max(loss_window) - min(loss_window)) / max(
            abs(np.mean(loss_window)), 1e-12
        )
        converged = (
            relative_span < cfg["optimization_loss_tolerance"]
            and record["grad_norm"] < cfg["optimization_grad_tolerance"]
            and record["settled"]
        )
        convergence_hits = convergence_hits + 1 if converged else 0
        if convergence_hits >= cfg["optimization_patience"]:
            print(
                f"optimization convergence reached at step {step}: "
                f"relative loss span={relative_span:.3e}"
            )
            break


  1%|          | 1/150 [00:27<1:08:06, 27.42s/it]

step=0000 loss=0.838148 Cd/disk=0.83815 RCS/disk=1.26036 chord=0.7000 grad=9.658e+01 adj=182 r_adj=6.99e-03 r_flow=3.66e-06 settle=4000 du=1.36e-03 dCd=4.22e-04 Ma_max=0.076 time=27.4s


  1%|▏         | 2/150 [00:56<1:09:44, 28.27s/it]

step=0001 loss=0.837771 Cd/disk=0.83777 RCS/disk=1.25770 chord=0.6995 grad=9.877e+01 adj=182 r_adj=6.99e-03 r_flow=3.42e-06 settle=1200 du=1.26e-03 dCd=1.22e-04 Ma_max=0.076 time=28.9s


  1%|▏         | 2/150 [01:25<1:45:59, 42.97s/it]


KeyboardInterrupt: 